Я хочу **по самому тексту запроса клиента** (его описанию того, что он разрабатывает) **извлечь максимально полное понимание** — что это за продукт, какие у него скрытые требования, какая бизнес-логика, а потом уже под это подобрать облачные услуги.

То есть я **не задаю вопросы**, а **анализирую его промпт** как NLP-задачу и выдаёте готовое решение с доп.фишками, которые **вытекают из анализа его же текста**.

## 🧩 Категория 1: Топология и география (поиск по словам)

Что ищете в тексте | Что это значит | Какую облачную фишку предложить
--- | --- | ---
«международный», «СНГ», «Европа», «Азия», «по всему миру» | Клиенты распределены глобально | **CDN с глобальным покрытием + Anycast IP + мультизонный балансировщик**
«регионы», «федеральный», «вся РФ», «Дальний Восток» | Нужна низкая задержка внутри РФ | **Распределённые зоны доступности (Москва + СПб + Екатеринбург) + geo-DNS**
«офлайн», «физический магазин», «касса», «торговый центр» | Есть on-prem устройства или гибрид | **VPN + Direct Connect + гибридное облако с роутингом**
«локальный», «городской», «район», «доставка 1 час» | Узкая география, но критична скорость | **Одна зона доступности с максимальными сетевыми лимитами + Edge-сервер рядом**
«мобильное приложение», «ios», «android», «app» | API-нагрузка, часто маленькие запросы | **API Gateway + кэширование ответов + авторейд на бэке**

---

## 🧩 Категория 2: Тип данных (поиск по словам)

Что ищете в тексте | Что это значит | Какую облачную фишку предложить
--- | --- | ---
«фото», «картинка», «изображение», «галерея», «медиа» | Много тяжёлых статических файлов | **S3-совместимое хранилище под медиа + CDN с префетчем + производные изображения (webp)**
«видео», «потоковое», «курсы», «лекции», «вебинар» | Высокая пропускная способность, нужен видеохостинг | **Объектное хранилище + видеотранскодинг как сервис + медиасервер (например, Nimble)**
«3d», «модель», «stl», «объект», «c4d», «vr» | Тяжёлые бинарные файлы, нужен просмотрщик | **GPU-инстанс под рендеринг + CDN + предвью-генерация**
«текст», «статья», «блог», «новость», «пост» | Интенсивность на запись, но лёгкий вес | **БД с высокой скоростью записи + Redis под горячие статьи + полнотекстовый поиск**
«аналитика», «отчёт», «дашборд», «метрики», «статистика» | OLAP-нагрузка, агрегации | **Отдельная read replica БД + колоночная БД (ClickHouse как сервис) + Snowflake-подобный кэш**

---

## 🧩 Категория 3: Взаимодействие с пользователем (поиск по словам)

Что ищете в тексте | Что это значит | Какую облачную фишку предложить
--- | --- | ---
«чат», «мессенджер», «realtime», «push», «уведомления» | Websockets / SSE, много одновременных соединений | **Выделенный балансировщик с поддержкой sticky sessions + масштабирование по числу соединений**
«гугл карты», «карта», «геолокация», «трекер» | Тяжёлые JS-библиотеки + запросы к тайловому серверу | **Проксирование тайлов с кэшированием + увеличенный memory для браузера на сервере**
«комментарии», «отзывы», «рейтинг», «лайки» | Высокая частота мелких запросов на запись | **Буферизированная запись через очередь (Kafka/RabbitMQ) + асинхронное обновление**
«корзина», «оформление», «оплата», «чек», «покупка» | Транзакционная целостность, деньги | **БД с двухфазным коммитом + бэкап до/после операции + WAF на оплату + PCI DSS контур**
«поиск», «фильтр», «каталог», «подбор» | Полнотекстовый поиск или фасеты | **Elasticsearch/OpenSearch как сервис + отдельный поисковый инстанс + индексация через очереди**

---

## 🧩 Категория 4: Интеграции и внешние системы (поиск по словам)

Что ищете в тексте | Что это значит | Какую облачную фишку предложить
--- | --- | ---
«1с», «склад», «умя», «мойсклад», «sap», «erp» | Внешние выгрузки с чёткой периодичностью | **Выделенный воркер для выгрузок + очереди сообщений + синхронный/асинхронный шлюз**
«сдэк», «боксберри», «почта», «доставка», «логистика» | API-запросы к внешним трекерам, нужны повторы | **Менеджер очередей с dead letter + ретраи + мониторинг ошибок интеграции**
«смс», «email», «рассылка» | Фоновые задачи, возможны спам-риски | **Dedicated IP для отправки + контроль репутации + rate limiter на исходящие**
«paypal», «юкасса», «найди меня», «эквайринг» | Чувствительные API, требуется безопасность | **Web Application Firewall + ограничение по IP + сетевые политики на платёжный модуль**
«telegram», «whatsapp», «viber», «бот» | Постоянные webhook-соединения, мгновенность | **Выделенный эндпоинт с авторейдом + сервер без буферизации ответов**

---

## 🧩 Категория 5: Поведенческие паттерны (поиск по косвенным признакам)

Что ищете в тексте | Что это значит | Какую облачную фишку предложить
--- | --- | ---
есть слово «секунда», «мгновенно», «реалтайм», «живой», «онлайн» | Жесткие требования к latency | **High-frequency CPU (3+ ГГц) + NVMe с IOPS > 50k + сеть с PPS > 1 млн**
есть слово «ночью», «утром», «в час пик», «вечером» | Неравномерная нагрузка по времени | **Автомасштабирование по расписанию + burst-инстансы на пик + спотовые ночью**
есть «миллион», «сотни тысяч», «высоконагруженный», «масштабируется» | Предсказуемо большая нагрузка | **Kubernetes с HPA + очереди + отдельный кластер БД с шардированием**
есть «бесплатно», «францум», «freemium», «демо» | Есть публичный неавторизованный доступ | **DDoS-защита + rate limiting на IP + отдельный пул для гостей + кэш**
есть «секретно», «безопасно», «конфиденциально» | Работа с персональными данными | **Шифрование дисков + выделенный физический сервер (bare metal) + закрытые сети**

---

## 🧩 Категория 6: Жизненный цикл данных (поиск по глаголам и временным указателям)

Что ищете в тексте | Что это значит | Какую облачную фишку предложить
--- | --- | ---
«архив», «старые заказы», «логи за год», «история» | Холодные данные редко используются | **Tiered storage (горячий NVMe → холодный S3 → ледниковый архив) + политики истечения**
«бэкап», «восстановление», «бекап», «точка отказа» | Требования к RPO и RTO | **Расписание снапшотов каждые 4 часа + репликация в другую зону + выделенный restore-полигон**
«лог», «логировать», «debug», «трассировка» | Нужна отладка без остановки | **Managed Loki или ELK + сбор логов с фильтрацией + долгосрочное хранение в S3**
«откатить», «версия», «роллбэк», «canary» | Частые выкладки, нужна безопасная доставка | **Blue-green deployment + canary-релизы + снэпшот перед деплоем**
«очистка», «удалить старые», «ttl», «устаревает» | Автоматическая смерть данных | **Idempotent-скрипты как сервис + TTL на объектах S3 + expiry на Redis**

---

## 🧩 Категория 7: Технологический стек (поиск по движкам и типовым словам)

Что ищете в тексте | Что это значит | Какую облачную фишку предложить
--- | --- | ---
«wordpress», «opencart», «magento», «shopify» | Монолит с тяжёлым PHP | **PHP-FPM с выделенным пулом + Redis для сессий + кэш страниц (Varnish)**
«django», «rails», «laravel», «spring» | MVC-фреймворк, стандартный рантайм | **Managed база данных + автоскейлинг воркеров очередей + отдельный assets-сервер**
«react», «vue», «next», «nuxt», «ssr» | Нужен рендеринг на сервере (SSR) | **Node.js инстанс с увеличенным memory + CDN на статику + выделенный API-шлюз**
«серверлесс», «lambda», «function», «без сервера» | Событийная архитектура | **FaaS как сервис + API Gateway + очередь как триггер + облачное хранилище состояний**
«docker», «kubernetes», «k8s», «контейнер» | Контейнеризация | **Managed K8s с Cluster Autoscaler + Container Registry + сервис-меш (опционально)**

---

## 🧩 Категория 8: Безопасность (явные и скрытые требования)

Что ищете в тексте | Что это значит | Какую облачную фишку предложить
--- | --- | ---
«ДК», «Договор», «подпись», «юридический» | Легальные требования к сохранности документов | **WORM-хранилище (write once read many) + аудит доступа + шифрование на уровне tenant**
«карта», «платёж», «cvv», «паспорт» | Прямая работа с платёжными данными (не агрегатор) | **PCI DSS контур + изолированные ноды + HSM-модуль (если есть у провайдера)**
«админка», «управление», «панель», «модерация» | Отдельный высокопривилегированный контур | **Отдельная подсеть + MFA на доступ + jump host + аудит всех команд**
«api для партнёров», «открытое апи», «вебхук для всех» | Публичный API, потенциальные атаки | **API gateway с лимитами + WAF + алерты на странные паттерны + квотирование**
«тестирование», «стенд», «dev», «staging» | Нужно отделить бой от разработки | **Полная копия инфраструктуры с дешёвыми ресурсами + теневая база + snapshot прода**

---

## Как это применить на практике (без вопросов клиенту)

Ты делаешь **простую эвристику**:

1. Берёте промпт клиента (например: *«Делаю видеоплатформу с курсами, пользователи по всей России, нужно, чтобы видео открывалось мгновенно, есть чат в реальном времени и оплата через карту»*)
2. Проходите по таблицам выше и собираете **все совпадения ключевых слов**.
3. Формируете итоговый список облачных фишек **автоматически**:

> *На основе вашего описания:*
> - «видео» → S3 + CDN + транскодинг
> - «по всей России» → geo-балансировка + зоны Мск/СПб/Екб
> - «мгновенно» → High-frequency CPU + NVMe + сеть с PPS
> - «чат в реальном времени» → Балансировщик с sticky + масштабирование по соединениям
> - «оплата через карту» → Изолированный платёжный контур + WAF + шифрование дисков
>
> *Итоговый стек:* […список облачных услуг…]

Таким образом вы не спрашиваете — вы **извлекаете** скрытые требования из его же текста.


# 🔍 Категория 1: Анализ ГЛАГОЛОВ (что делает продукт)

Глагол в промпте | Что это означает | Какие услуги облака цеплять
--- | --- | ---
«продаю», «торгую», «принимаю оплату» | E-commerce / Retail | Балансировщик + WAF + PCI DSS-контур + БД с транзакциями + Redis для корзины
«храню», «сохраняю», «архивирую» | Хранилище документов / файлов | Объектное S3 + Glacier-архив + бэкапы + шифрование покоя + версионирование
«обрабатываю», «парсю», «агрегирую» | ETL / Аналитика | Batch-инстансы + очереди сообщений + ClickHouse + выделенные воркеры
«общаюсь», «чат», «звоню» | Коммуникация / VoIP / Messaging | WebRTC-серверы + выделенные сокеты + Redis Pub/Sub + медиасерверы
«смотрю», «видео», «трансляция» | Streaming / Media | Видеотранскодер + CDN + S3 под исходники + сеть с большим egress
«слушаю», «музыка», «аудио», «подкаст» | Аудиосервис | Объектное хранилище + аудиоплеер-бэкенд + CDN + пресеты битрейта
«рисую», «рендерю», «3d», «графика» | Визуализация / CG | GPU-инстансы + высокоскоростной диск + VDI + объектное под ассеты
«управляю», «devices», «iot», «датчики» | IoT / M2M | MQTT-брокер + БД временных рядов (TimescaleDB) + миллионные соединения
«бронирую», «таймер», «занято» | Билеты / Залы / Отели | Redis с TTL + транзакционная БД + очередь освобождения + снэпшоты
«перевожу», «конвертирую», «формат» | Транскодинг / Конвертация | Spot-инстансы + очереди + S3 как буфер + шаблоны конвертации

---

## 🔍 Категория 2: Анализ СУЩЕСТВИТЕЛЬНЫХ (с чем работает продукт)

Существительное | Что за домен | Какие услуги
--- | --- | ---
«заказ», «чек», «счет», «инвойс» | Финансовые документы | БД с ACID + бэкапы + шифрование + аудит всех операций
«клиент», «пользователь», «юзер», «аккаунт» | Identity Management | База пользователей + Redis сессий + MFA + Federation Gateway
«товар», «склад», «инвентарь» | Инвентаризация / WMS | БД с конкурентным обновлением + очереди + периодические выгрузки
«курс», «урок», «тренинг», «лекция» | EdTech / LMS | CDN под видео + S3 под материалы + API для прогресса + вебхуки
«сообщение», «пост», «коммент», «лайк» | Социальное / UGC | БД с высокой скоростью записи + кэш + очередь + фоновые аггрегаторы
«событие», «ивент», «встреча», «конференция» | Event-менеджмент | Бронирование через Redis + рассылки + тикеты с генерацией PDF
«медицина», «врач», «запись», «анализ» | HealthTech / MedTech | Шифрование персональных данных + аудит доступа + соответствие 152-ФЗ
«логистика», «груз», «маршрут», «трек» | Logistics / Fleet | База геоданных (PostGIS) + очереди на обновление треков + websockets
«нейросеть», «ai», «ml», «модель» | ML / AI | GPU-инстансы + объектное под датасеты + деплой моделей через TorchServe
«биллинг», «тарификация», «списание» | Billing / Rating | БД с двухфазным коммитом + фоновые расчёты + аудит всех движений

---

## 🔍 Категория 3: Анализ ПРИЛАГАТЕЛЬНЫХ (неявные требования)

Прилагательное | Скрытая потребность | Какие услуги
--- | --- | ---
«быстрый», «мгновенный», «живой» | Low latency (< 50 мс) | High-frequency CPU + NVMe + сеть с PPS > 500k + Redis-кэш
«надёжный», «безотказный», «24/7» | High Availability | Репликация в 2+ зоны + автоматический failover + SLA 99.95%+
«безопасный», «защищённый», «приватный» | Compliance / Security | Шифрование дисков + VPC + WAF + аудит логов + VPN
«дешёвый», «бюджетный», «экономный» | Cost-sensitive | Спот-инстансы + холодное S3 + автоотключение dev-стендов + Reserved Instances
«масштабируемый», «на много пользователей» | Horizontal scaling | K8s + HPA + очереди + шардирование БД + CDN
«тяжёлый», «большой», «гигабайты» | Big data / Large objects | Объектное хранилище + многопоточная загрузка + параллельное чанкирование
«интерактивный», «реалтайм», «онлайн» | Real-time interaction | Websockets + серверная событийная архитектура + очереди с низкой задержкой
«автоматический», «без рук», «скрипт» | Automation / Scheduled jobs | Крон как сервис + serverless functions + workflow-оркестратор
«красивый», «визуальный», «графика» | Rich UI / много ассетов | CDN + картинки с преобразованием форматов + lazy-loading готовый
«умный», «рекомендует», «подбирает» | Recommender system | Онлайн-инференс + векторная БД + GPU под ретрайн

---

## 🔍 Категория 4: Анализ ЧИСЕЛ и ОБЪЁМОВ (если их упомянул)

Фраза с числом | Оценка нагрузки | Какие услуги
--- | --- | ---
«тысячи», «миллионы», «сотни тысяч» | High-load | K8s с автоскейлером + шардирование БД + очереди + CDN + read replicas
«секунды», «минуты», «часы» (время ответа) | Строгие timeouts | Dedicated CPU + NVMe с гарантированным IOPS + SLA на сеть
«терабайты», «петабайты» | Big Data | S3 + Glacier + распределённые вычисления (Spark как сервис)
«10 000+ пользователей одновременно» | Concurrency | Балансировщик + sticky sessions + Redis под сессии + форки воркеров
«1000 запросов в секунду» | RPS-нагрузка | API Gateway + rate limiting + кэш + горизонтальное масштабирование
«99.9%», «99.99%» | Availability SLA | Multi-AZ + автоматический failover + health checks + резервные пути
«24 часа», «круглосуточно» | No downtime allowed | Живая миграция + rolling updates + blue-green деплой

---

## 🔍 Категория 5: Анализ ИНТЕГРАЦИЙ (о чём продукт говорит в связке)

Упоминание | С кем интегрируется | Какие услуги
--- | --- | ---
«1с», «склад», «умя», «мойсклад» | Учётные системы | Выделенные очереди + периодические воркеры + выгрузка в CSV/XML/JSON
«сдэк», «boxberry», «почта россии» | Логистические API | Очереди с ретраями + мониторинг ответов + dead letter queue
«телеграм», «whatsapp», «viber» | Мессенджеры | Webhook-эндпоинты + очереди сообщений + статические IP для белых списков
«yandex», «google», «maps», «карты» | Геосервисы | Проксирование тайлов + кэширование запросов + балансировка по картам
«paypal», «stripe», «юkappa», «cloudpayments» | Платёжные шлюзы | Изолированный сейф-контур + WAF + валидация подписей запросов
«kafka», «rabbit», «redis pubsub» | Брокеры сообщений | Managed очередь + мониторинг лагов + масштабирование консьюмеров
«oauth», «sso», «ldap», «active directory» | Единая аутентификация | Identity provider + SSO шлюз + токены + сессии
«s3», «gcs», «azure blob» | Облачное хранилище | Шлюз с поддержкой multipart upload + объектная репликация

---

## 🔍 Категория 6: Анализ ТОНАЛЬНОСТИ и СТИЛЯ (характер продукта)

Стиль фразы | О чём говорит | Какие услуги
--- | --- | ---
«мне нужно быстро стартануть», «MVP», «прототип» | Стартап-спринт | Managed сервисы + PaaS (Heroku-подобный) + автосоздание БД + готовые шаблоны
«должно выдержать черную пятницу» | Циклические пики | Автомасштабирование + burst CPU + резервные ёмкости по запросу
«я только учусь», «пет-проект», «для себя» | Некоммерческая нагрузка | Бесплатный тир (если есть) + спотовые + минимальные инстансы
«наше legacy», «старая система», «миграция» | Перенос с on-prem | VPN + Direct Connect + конвертация дисков + тестовый прогон
«enterprise», «корпоративный», «крупный бизнес» | Требования к безопасности/поддержке | Выделенные ноды + 24/7 поддержка + соглашение об уровне сервиса + аудит
«мы уже растем», «масштаб проблемы» | Текущая боль от роста | K8s + очереди + реплики БД + отдельный читающий инстанс
«бюджет ограничен», «стартап», «без денег» | Price-sensitive | Споты + резервирование на год + cold tier + автостоп неактивных VM

---

## 🔍 Категория 7: Анализ НАЛИЧИЯ готовых платформ (узнаваемый домен)

Фраза | Намёк на готовый продукт | Какие услуги
--- | --- | ---
«интернет-магазин», «корзина», «каталог», «оплата» | E-commerce | PHP/Node.js хостинг + MySQL/PostgreSQL + Redis + CDN + WAF + поиск
«доска объявлений», «авито», «юла», «чат» | Classifieds | Полнотекстовый поиск + модерация + очереди + websockets
«агрегатор», «сравнение цен», «парсинг» | Price aggregator | Spot-воркеры + очереди парсинга + прокси-менеджер + БД с версионированием
«социальная сеть», «лента», «друзья», «подписка» | Social network | Newsfeed-генерация + графовая БД (опционально) + push-уведомления
«crm», «клиенты», «сделки», «воронка» | CRM | БД с тегами + поиск + права доступа + аудит + экспорт/импорт
«биржа», «обмен», «торги», «ордер» | Trading / Marketplace | БД с атомарностью + очередь + транзакции + вебсокеты на цены
«сбор заказов», «курьер», «ресторан», «доставка» | Delivery / Foodtech | Гео БД + трекинг в реальном времени + назначение курьеров + push

---

## 🔍 Категория 8: Анализ ФОРМАТА ВЫВОДА (что продукт генерирует)

Что создаёт продукт | Требование к облаку | Какие услуги
--- | --- | ---
«pdf», «отчёт», «документ», «счёт» | Генерация файлов | Временное хранилище + очереди генерации + S3 для хранения результата
«график», «диаграмма», «дашборд» | Визуализация данных | In-memory агрегация + кэш + CDN под виджеты
«email», «письмо», «уведомление» | Email-рассылки | Dedicated SMTP + контроль репутации + очереди + шаблонизатор
«sms», «пуш», «уведомление на телефон» | Push/SMS | Шлюз с rate limit + очереди + лог доставки
«api», «эндпоинт», «webhook», «rest» | API-сервис | API Gateway + rate limit + мониторинг + OpenAPI-спека
«голос», «звонок», «ivr» | VoIP / IVR | Медиасервер + запись разговоров + транскрипция + CDR
«архив», «zip», «дамп» | Упаковка данных | Сжатие как сервис + потоковая выгрузка в S3

---

## Как это агрегировать в итоговый результат

Пример клиентского промпта:
> *«Делаю платформу для онлайн-курсов с видео, тысяча пользователей одновременно, нужен быстрый отклик»*

Проходите по таблицам → собираете **признаки**:
- «видео» → Category 1 (смотрю) → видеотранскодер + CDN + S3
- «курсы» → Category 2 (курс) → CDN + S3 + API прогресса
- «тысяча одновременно» → Category 4 (concurrency) → балансировщик + sticky + Redis сессии
- «быстрый отклик» → Category 3 (быстрый) → high-frequency CPU + NVMe

Вывод:
> *На основе вашего описания: EdTech-платформа с видео-контентом и гарантированной интерактивностью. Рекомендуем: Managed K8s, S3 для видео + CDN, Redis для сессий, High-frequency CPU, балансировщик с автоскейлом.*

**Ни одного вопроса клиенту не задано.**

Хотите — следующим шагом сделаю **матрицу «промпт-токен → услуга из 200»** для автоподбора?

----

**50 нестандартных улучшений**, которые вы можете предложить **автоматически** на основе анализа его промпта. Они неочевидны, но бьют в реальные боли.

## 💥 Категория 1: Антихрупкость (то, что сломается, но он не знает)

Нестандартное улучшение | Когда предлагать | Какая услуга облака
--- | --- | ---
**«Троттлинг корзины при пике»** | В промпте есть «распродажа», «скидка», «акция» | Rate limiter + очереди + отказоустойчивая запись в БД
**«Фейковый эндпоинт для DDoS-приманки»** | Магазин, банк, любой публичный сервис | HoneyPot + отдельный IP + алертинг
**«Медленный режим для роботов»** | Есть «парсинг», «аггрегатор» или «сравнение цен» | Задержка ответов для не-браузеров + троттлинг по User-Agent
**«Снапшот до каждой миграции БД»** | Есть «разработка», «активное обновление» | Автоматический пре-миграционный снэпшот с тегом даты
**«Автоматический откат по RPS-аномалии»** | Есть «высокая нагрузка», «миллионы» | Canary + метрики + автоматический роллбэк
**«Изоляция записи логов на отдельный диск»** | Есть «логи», «debug», «аудит» (всегда) | Отдельный недорогой том + logrotate
**«Тестовая распродажа на копии бо€€€я»** | E-commerce, билеты, бронирование | Клон инфраструктуры на 2 часа + нагрузочное тестирование
**«Холодный резервный кэш (если Redis упал)»** | Зависит от Redis (корзина, сессии, таймеры) | Второй Redis в режиме replica + автоматический failover
**«Чёрный ход в БД для админа»** | Есть «админка», «модерация» | PgBouncer с отдельным пулом + jump host
**«Очередь с приоритетами (платёж всегда первый)»** | Есть «оплата», «корзина», «чек» | RabbitMQ с приоритетами + dead letter на второстепенные задачи

---

## 🧠 Категория 2: Поведенческая телепатия (он ещё не знает, что так сделает)

Нестандартное улучшение | Когда предлагать | Какая услуга облака
--- | --- | ---
**«Горячий товар — в RAM заранее»** | Каталог, магазин, агрегатор | Предзагрузка top-100 SKU в Redis при старте
**«Предпросмотр тяжелых страниц через очередь»** | Видео, 3D, большие изображения | Асинхронная генерация превью через воркеры
**«Ленивая загрузка истории заказов»** | Много заказов, пользовательская история | Пагинация с курсором + архив на S3
**«Кеш "пользователь только что зарегистрировался"»** | Есть регистрация, аккаунты | Короткий TTL (5 минут) + отдельный Redis
**«Генерация PDF-счета в фоне (даже если не попросили)»** | E-commerce, билеты, документы | Предгенерация с сохранением на S3 + ссылка
**«Стартовый прогресс-бар на бэкенде (симуляция работы)»** | Долгие операции («обработка», «парсинг») | Serverless функция на SSE
**«Автосохранение черновика корзины»** | Любой магазин с корзиной | Redis с долгим TTL + периодический дамп
**«Если пользователь вернулся — восстановить сессию в 0 мс»** | Есть пользователи, сессии, авторизация | Предзагрузка сессии в L1 cache на балансировщике
**«Адаптивный размер картинки через URL-параметр»** | Много картинок, галерея | Image resizer как сервис + CDN с кэшем вариантов
**«Ленивый расчёт доставки (пока не введён адрес)»** | Доставка, логистика, магазин | Отложенный триггер + кэш расчётов по zip-коду

---

## ⚡ Категория 3: Скрытые узкие места (которые убьют его через 3 месяца)

Нестандартное улучшение | Когда предлагать | Какая услуга облака
--- | --- | ---
**«БД начинает тормозить на 1 млн записей — вот починка»** | Есть «расту», «пользователи», «заказы» | Партиционирование таблиц + архивация старых
**«Внезапно умрёт соединение с внешним API»** | Есть «сдэк», «paypal», «телеграм» | Circuit breaker + retry с экспоненциальной задержкой
**«Нагрузка на запись убьёт read реплику»** | Есть «чат», «лайки», «комментарии» | Выделенный writer + несколько readers в балансе
**«Redis съест всю память и упадёт»** | Есть «сессии», «корзина», «кэш» | maxmemory-policy allkeys-lru + алерт на 80%
**«Сеть упрётся в PPS, не в трафик»** | Много маленьких запросов, чат, API | High-PPS инстанс + проверка лимитов
**«Логи сожрут всё место за неделю»** | Есть «логировать», «debug», «аудит» | Centralized logging с ротацией + retention 14 дней
**«Индексы в БД перестанут влезать в память»** | Крупная БД, много запросов | Partial индексы + блум-фильтры + векторизация
**«Таймзона пользователя убьёт кэш»** | Глобальная аудитория, «по всему миру» | Кэш с учётом таймзоны + разделение по региону
**«Долгие запросы отчётов заблокируют продажи»** | Есть «отчёты», «аналитика», «дашборды» | Read replica под отчёты + kill долгих запросов
**«Ночной бэкап сожрёт весь IOPS»** | Любой сервис с данными | Throttle бэкапов + отдельный диск под бэкап

---

## 🪄 Категория 4: UX-магия (клиент не просил, но пользователь полюбит)

Нестандартное улучшение | Когда предлагать | Какая услуга облака
--- | --- | ---
**«Имитация мгновенного ответа, пока идёт расчёт»** | Долгие операции, поиск, рекомендации | Оптимистичный UI state + фоновая очередь
**«Предзагрузка второй страницы каталога»** | E-commerce, маркетплейс | HTTP/2 push или Link prefetch на бэкенде
**«Если видео долго грузится — отдать 360p сначала»** | Видеоплатформа, курсы | Адаптивный битрейт + трансмуксеринг
**«При ошибке оплаты — сохранить корзину и показать другие способы»** | Платёжный шлюз | Состояние корзины в Redis + fallback методы
**«Пользователь ушёл — отправить напоминалку через 15 минут»** | Корзина, бронирование, незаконченная регистрация | Отложенные задачи + web push
**«Поиск с опечатками "как в гугле"»** | Поиск, каталог | Elasticsearch с fuzzy search + синонимайзер
**«Автокомплит поиска за 10 мс (даже при 10 млн товаров)»** | Большой каталог | Edge-cached автокомплит + урезанный индекс
**«Генерация meta-тегов для соцсетей на лету»** | Контентный сервис, блог, магазин | Serverless рендер OpenGraph + CDN
**«Показ "этот товар смотрят сейчас" (эффект толпы)»** | E-commerce, билеты | WebSocket + счётчик просмотров в Redis
**«Режим "только чтение" при перегрузке»** | Любой сервис с пиками | Graceful degradation через балансировщик

---

## 🔐 Категория 5: Параноидальная безопасность (про которую он забыл)

Нестандартное улучшение | Когда предлагать | Какая услуга облака
--- | --- | ---
**«А если админский пароль украдут — у нас есть второй фактор на входе в VM»** | Есть «админка», «управление», любые права | MFA на SSH + jump host + аудит
**«Конфигурация сети так, что даже при взломе бэкенда не достать платёжную БД»** | Платежи, карты, персональные данные | Нейтральная сеть + security groups + VPC peering
**«Логи запросов к БД хранятся 3 месяца (для расследования)»** | Финансы, медицина, юристы | Аудит PostgreSQL + отдельное хранилище логов
**«Если WAF не сработал — есть резервный allowlist IP»** | Публичный сервис | IP whitelisting на слой выше + fallback ACL
**«Снапшот диска каждые 10 минут перед акцией»** | E-commerce, распродажа | Frequent snapshots + автоматическое удаление старых
**«Выделенный пользователь в БД только для SELECT»** | Есть отчёты, дашборды, аналитика | Read-only роль + отдельный пул соединений
**«Токен сессии умирает при смене пароля (все устройства)»** | Пользователи, аккаунты | Инвалидация JWT через Redis blocklist
**«Архив старых заказов на отдельном зашифрованном томе»** | Долгие данные, compliance | LUKS + S3 с SSE-C + ключи отдельно
**«Rate limit НЕ по IP, а по поведению (например, слишком быстрый клик)»** | Публичный API, магазин | Адаптивный лимитер + sliding window
**«Если DDoS — включить "капчу для всех новых сессий"»** | Любой публичный сервис | Автоматический challenge-mode на балансировщике

---

## 🧩 Категория 6: Экономия, о которой он не догадывается

Нестандартное улучшение | Когда предлагать | Какая услуга облака
--- | --- | ---
**«Спотовые инстансы под парсинг — экономия 70%»** | Есть «парсю», «анализирую», «фоновый» | Spot fleet + авто-перезапуск при вытеснении
**«Ночью уменьшаем кластер вдвое»** | Нет ночной активности | Cron + HPA с расписанием
**«Логи через 7 дней уходят в S3 Glacier и платят копейки»** | Много логов, долгое хранение | Lifecycle policy S3 → Glacier
**«Кэш в памяти умирает, если не используется 2 часа»** | Любой кэш Redis/Memcached | TTL + volatile-lru eviction
**«Тестовая среда живёт только с 9 до 20 по будням»** | Есть «dev», «staging», «тестирование» | Автостоп VM + schedule start/stop
**«Реплика БД на споте для отчётов (дешевле в 3 раза)»** | Есть отчёты, аналитика, дашборды | Spot-инстанс с репликой + выключение при нестабильности
**«Объектное хранилище вместо диска под картинки»** | Много медиа, картинок | S3 + CDN вместо EBS/блочных дисков (дешевле в разы)
**«Зарезервированные инстансы на базу данных на год»** | БД работает 24/7, нет гибкости | Reserved Instances + конвертируемые RI
**«Очистка неиспользуемых снапшотов (их легко забыть)»** | Любой сервис с бэкапами | Автоматический find + delete старых снапшотов
**«Если VM idle > 2 часов — подскажем заменить на серверлесс»** | Неравномерная нагрузка | FaaS + API Gateway

---

## Как это упаковать в ответ клиенту

Вместо сухого списка услуг вы говорите (автоматически генерируете):

> *«Смотрю ваше описание. Помимо базовых мощностей, вот **3 неочевидные вещи**, которые вас спасут:*
> 1. *Вы пишете про "парсинг" — значит вы упрётесь в счётчик одновременных соединений, а не в трафик. Добавим инстанс с high PPS.*
> 2. *"Распродажа" — автоматический снапшот БД за 5 минут до её начала. Откат за 10 секунд.*
> 3. *"Бюджет ограничен" — ночные спотовые инстансы под отчёты сэкономит 60%.»*

Клиент чувствует, что вы **прочитали его мысли** и знаете об облаке больше, чем просто «диски и CPU».

